# GoalMind: xG, gol e forza delle squadre

**Domanda:** come cambia una stima della probabilità di segnare almeno un gol con i minuti giocati e con i gol subiti dall'avversaria?

Usiamo due CSV storici, puliamo le osservazioni, calcoliamo Elo e confrontiamo gol e xG. Le celle importano il codice del progetto. Le stime sono scenari esplorativi: non sono previsioni calibrate.

Il CSV dei giocatori non documenta stagione/data di estrazione; lo storico delle partite termina il 1° giugno 2025. La provenienza originale dei file e il loro allineamento temporale devono essere completati prima della consegna.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Il notebook può essere aperto dalla radice o da una sua sottodirectory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'main.py').is_file() and (p / 'import').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Aprire il notebook nella cartella del repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from main import load_application

application = load_application()
pd.Series(application.load_report, name='Conteggio')

## 1. Input e pulizia

I valori `-` sono mancanti, non zeri. La chiave (nome, squadra) conserva gli omonimi. Gli alias collegano nomi diversi della stessa squadra senza una ricerca approssimativa. Le partite sono ordinate per data/orario; punteggi mancanti, negativi o non interi sono scartati e contati. Le date di inizio/fine sono inclusive.

Con i CSV originali ci aspettiamo 1.533 osservazioni e 28 tassi xG/90 mancanti. I tre risultati mancanti nello storico completo sono precedenti al filtro predefinito.

In [ ]:
players = application.player_table()
print('Squadre senza storico:', application.unmatched_player_teams())
print('Ultima partita:', application.snapshot_date.date())
print('Finestra difensiva: dopo', application.defense_start_date.date())
display(players.head())
display(players.loc[players.Name.eq('Juan Cruz')])
display(players.isna().sum().to_frame('Mancanti'))

## 2. Analisi descrittiva: xG e gol totali

Confrontiamo il totale xG del CSV con il totale dei gol. Confrontare xG/90 e gol totali introdurrebbe una differenza di esposizione. Pearson descrive l'associazione nello snapshot; non valida il modello sulle partite future. I minuti giocati influiscono su entrambi i totali, e le osservazioni non sono necessariamente indipendenti. Il p-value viene riportato come statistica descrittiva, senza conclusioni causali.

In [ ]:
report = application.statistics()
pd.Series(report, name='Statistiche descrittive')

## 3. Elo: forza complessiva

Ogni squadra parte da 1.500; K=25 e vantaggio casa=40. Il risultato vale 1, 0,5 o 0. L'aggiornamento è simmetrico e conserva la somma dei rating. La logistica Elo restituisce il punteggio atteso, non una probabilità di vittoria separata dal pareggio.

Classifichiamo squadre attive di una divisione: campionati scollegati non hanno una scala comune calibrata. L'Elo non viene usato come misura della difesa.

In [ ]:
division = 'I1'  # Serie A; provare E0, D1, SP1 o F1
application.team_table(division).head(10)

## 4. Visualizzazione

I grafici vengono creati dal modulo condiviso: gol contro xG (giocatori con almeno 500 minuti), distribuzione del rating e top Elo della divisione scelta. La diagonale confronta gol e xG, non indica accuratezza di previsione.

In [ ]:
figures = application.visualize_data(show=False, division=division)
for figure in figures:
    display(figure)
    plt.close(figure)

## 5. Scenario di gol: intensità e minuti

Assumiamo un conteggio dei gol di Poisson con intensità costante:

$$\lambda = (xG/90)\,(m/90)\,d,\qquad P(G\geq 1)=1-e^{-\lambda}.$$

`d=1` è la base. L'aggiustamento descrittivo usa i gol subiti per partita dall'avversaria divisi per la media dei gol per squadra-partita nel suo campionato, nella stessa finestra di 365 giorni. Richiede almeno cinque partite e squadre nella stessa ultima divisione osservata.

La probabilità è condizionata ai minuti realmente giocati. Il fattore non corregge forza del calendario o casa/trasferta. Non si usano punteggi arbitrari di ruolo/tackle/rating.

In [ ]:
player_name = 'Mohamed Salah'
opponent_name = 'Arsenal'
player = application.find_player(player_name)
opponent = application.find_team(opponent_name)
print('xG/90:', player.xgperninety)
print('Fattore difensivo:', round(opponent.defensive_factor(), 3))
scenarios = pd.DataFrame({
    'Minutes': [30, 60, 90],
    'Base_probability': [player.prob_goal(m) for m in [30, 60, 90]],
    'Adjusted_probability': [application.score_probability(player_name, opponent_name, m)
                             for m in [30, 60, 90]],
})
scenarios.style.format({'Base_probability': '{:.1%}', 'Adjusted_probability': '{:.1%}'})

## 6. Output e limiti

Esportiamo risultati in CSV e PNG. Il modello non è stato validato su partite future: le statistiche aggregate del giocatore non sono datate e possono includere partite successive alla situazione che vorremmo prevedere. Per valutare previsioni servono osservazioni giocatore-partita, informazioni disponibili prima di ogni partita e una valutazione cronologica con misure di calibrazione.

La correlazione elevata nello snapshot non sostituisce questa verifica. La qualità del progetto sta in pulizia, organizzazione, calcoli corretti, grafici leggibili e ipotesi esplicite.

In [ ]:
output = ROOT / 'outputs'
output.mkdir(exist_ok=True)
players.to_csv(output / 'players.csv', index=False)
application.team_table(division).to_csv(output / 'teams.csv', index=False)
for figure in application.visualize_data(show=False, output_dir=output, division=division):
    plt.close(figure)
print('Esportati: players.csv, teams.csv e tre grafici PNG.')

## Per la discussione (circa 5 minuti)

1. Domanda e limiti dei dati.
2. Pulizia e omonimi, mostrando Juan Cruz.
3. Grafico gol/xG e distinzione tra associazione e previsione.
4. Elo e stima Poisson, mostrando come variano i minuti.
5. Esportazione, test e cosa servirebbe per validare il modello.

Approfondimento: `docs/CORREZIONI.md`. Eseguire `python -m unittest discover -s tests -v` dalla radice. Essere in grado di spiegare ogni parte del codice, incluse le correzioni realizzate con assistenza AI.